# Ordered Logistic Regression Results (FAIRˆ²) Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIRˆ²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library, following best practices for referencing dataset entities using their `@id` fields throughout.

### Dataset Source
The dataset is defined by a Croissant schema and can be found at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
We will use `mlcroissant` to load the dataset metadata and prepare for record inspection.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Access as an object, not as a dict

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's inspect all record sets defined in the dataset's Croissant package. For each record set, we will list their `@id` along with all available fields and columns and their respective `@id`s.

> **Note:** All entities (record sets, fields, columns) are referenced by their `@id`s.


In [ ]:
# List all record sets and their fields/columns by @id
print("Record Sets Overview:")
record_sets = []
for rs in dataset.record_sets:
    print(f"- Record Set Name: {rs.name}\n  @id: {rs.id}")
    field_ids = [field.id for field in rs.fields] if rs.fields else []
    column_ids = [col.id for col in rs.columns] if rs.columns else []
    if field_ids:
        print(f"  Fields (@id): {field_ids}")
    if column_ids:
        print(f"  Columns (@id): {column_ids}")
    record_sets.append(rs.id)
    print("")

print("Total record sets:", len(record_sets))

# For further steps, select the first available record set for demonstration
if record_sets:
    selected_record_set_id = record_sets[0]
    print(f"Selected record set for extraction: {selected_record_set_id}")

## 3. Data Extraction
We will load the records from each record set (referenced by their `@id`) into separate pandas DataFrames.

Pick one record set for analysis, and display its DataFrame columns to decide which field to use in the next stage.


In [ ]:
dataframes = {}

# Loop through discovered record sets and extract data
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if record_sets:
    print(f"DataFrame columns for record set '{selected_record_set_id}':")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (by its `@id`) from the chosen record set for filtering, normalization, and grouping. Please inspect the DataFrame column list above and replace `<numeric_field_id>` and `<group_field_id>` with valid column names as needed.

*Below, you'll see example code -- edit the placeholders if necessary to match actual available columns.*


In [ ]:
# Choose a field (by @id) present in the data for numeric analysis.
# Example: For regression results, commonly available fields: 'cr:log_likelihood', 'cr:coefficient', etc.
numeric_field = None
group_field = None
df = dataframes.get(selected_record_set_id)
if df is not None:
    # Find likely numeric and group fields
    for col in df.columns:
        if ('log_likelihood' in col or 'coefficient' in col or 'value' in col) and numeric_field is None and pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
        if ('ward' in col or 'region' in col or 'group' in col) and group_field is None:
            group_field = col
    # If none found, use the first column as fallback
    if numeric_field is None and len(df.columns)>0:
        numeric_field = df.columns[0]
    if group_field is None and len(df.columns)>1:
        group_field = df.columns[1]

if numeric_field:
    threshold = df[numeric_field].quantile(0.5) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0  # use median as reasonable threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())
    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (mean values):")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and, if available, how it varies by the selected group field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and (numeric_field in df.columns):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

if numeric_field and group_field and (group_field in df.columns):
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Using the `mlcroissant` library, we referenced all dataset elements by their `@id` fields, followed the Croissant schema, and explored records in a reproducible, standardized way. The EDA highlighted the main numeric patterns and possible group-wise differences in regression outputs for knowledge adoption in Northern Kenya, using genuine metadata-driven discovery.

**Next steps:**
- Try other record sets or fields by editing the notebook above.
- Combine numeric and categorical analysis for further policy and research insights.
